# EDA — Credit Scoring (Give Me Some Credit)

Objectif : comprendre les données avant toute modélisation.

**Règle de non-fuite de données** : le split train/test est effectué en tout premier, avant
tout nettoyage, imputation ou calcul de statistiques. Toute l'EDA ci-dessous porte uniquement
sur le train. Le test reste intouché jusqu'à l'évaluation finale du modèle.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42

## 0. Chargement des données brutes

Les données brutes ne sont jamais modifiées (`data/raw/`). La première colonne du CSV Kaggle
est un identifiant sans nom (`Unnamed: 0`) : on le renomme explicitement pour éviter toute
ambiguïté.

In [ ]:
RAW_PATH = "../data/raw/cs-training.csv"

df = pd.read_csv(RAW_PATH, index_col=0)
df.index.name = "id"

print(df.shape)
df.head()

In [ ]:
df.info()

## 1. Split train/test (AVANT tout nettoyage)

On sépare 80 % train / 20 % test, de façon **stratifiée** sur la cible `SeriousDlqin2yrs` pour
conserver la même proportion de défauts dans les deux ensembles (important vu le déséquilibre de
classes ~7 %). Le `random_state` est fixé pour la reproductibilité.

Ce découpage est fait maintenant, avant l'imputation des valeurs manquantes ou tout traitement
des outliers, pour éviter que des statistiques calculées sur l'ensemble complet (moyenne, médiane,
bornes de clipping...) ne "fuient" des informations du test vers le train.

In [ ]:
TARGET = "SeriousDlqin2yrs"

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df[TARGET],
    random_state=RANDOM_STATE,
)

print("Train :", train_df.shape, "- proportion défauts :", train_df[TARGET].mean().round(4))
print("Test  :", test_df.shape, "- proportion défauts :", test_df[TARGET].mean().round(4))

In [ ]:
import os

os.makedirs("../data/processed", exist_ok=True)
train_df.to_csv("../data/processed/train.csv")
test_df.to_csv("../data/processed/test.csv")

print("Sauvegardés dans data/processed/")

**À partir d'ici, toute l'EDA porte uniquement sur `train_df`.**

## 2. Proportion de défauts

### Conclusion :


In [ ]:
default_counts = train_df[TARGET].value_counts().sort_index()
default_pct = train_df[TARGET].value_counts(normalize=True).sort_index() * 100

summary = pd.DataFrame({"count": default_counts, "pct": default_pct.round(2)})
summary.index = summary.index.map({0: "Pas de défaut (0)", 1: "Défaut (1)"})
summary

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
train_df[TARGET].value_counts().sort_index().plot(kind="bar", ax=ax, color=["#4C72B0", "#C44E52"])
ax.set_xticklabels(["Pas de défaut (0)", "Défaut (1)"], rotation=0)
ax.set_ylabel("Nombre de clients")
ax.set_title("Distribution de la cible SeriousDlqin2yrs (train)")
plt.tight_layout()
plt.show()

## 3. Valeurs manquantes par colonne

### Conclusion :


In [ ]:
missing = train_df.isna().sum()
missing_pct = (train_df.isna().mean() * 100).round(2)

missing_summary = pd.DataFrame({"n_missing": missing, "pct_missing": missing_pct})
missing_summary = missing_summary[missing_summary["n_missing"] > 0].sort_values(
    "n_missing", ascending=False
)
missing_summary

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
sns.barplot(
    x=missing_summary["pct_missing"],
    y=missing_summary.index,
    ax=ax,
    color="#DD8452",
)
ax.set_xlabel("% de valeurs manquantes")
ax.set_title("Valeurs manquantes par colonne (train)")
plt.tight_layout()
plt.show()

## 4. Valeurs aberrantes

On regarde en particulier :
- `age` (valeurs impossibles, ex. 0)
- `DebtRatio` et `RevolvingUtilizationOfUnsecuredLines` (ratios qui devraient être bornés mais
  présentent parfois des valeurs extrêmes très supérieures à 1)
- les variables de retards de paiement (`NumberOfTime30-59DaysPastDueNotWorse`,
  `NumberOfTime60-89DaysPastDueNotWorse`, `NumberOfTimes90DaysLate`), connues dans ce dataset
  pour contenir des valeurs sentinelles **96 et 98** qui ne sont pas des comptes réels.

### Conclusion :


In [ ]:
cols_to_check = [
    "age",
    "DebtRatio",
    "RevolvingUtilizationOfUnsecuredLines",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]
train_df[cols_to_check].describe()

In [ ]:
# Âge : y a-t-il des valeurs impossibles (<=0) ?
print("Clients avec age <= 0 :", (train_df["age"] <= 0).sum())
print("Age min/max :", train_df["age"].min(), "/", train_df["age"].max())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sns.boxplot(y=train_df["age"], ax=axes[0], color="#4C72B0")
axes[0].set_title("age")

sns.boxplot(y=train_df["DebtRatio"], ax=axes[1], color="#55A868")
axes[1].set_title("DebtRatio")

sns.boxplot(y=train_df["RevolvingUtilizationOfUnsecuredLines"], ax=axes[2], color="#C44E52")
axes[2].set_title("RevolvingUtilizationOfUnsecuredLines")

plt.tight_layout()
plt.show()

In [ ]:
# Valeurs sentinelles 96 / 98 dans les compteurs de retards de paiement
late_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]

for col in late_cols:
    n_96 = (train_df[col] == 96).sum()
    n_98 = (train_df[col] == 98).sum()
    print(f"{col}: valeurs=96 -> {n_96} | valeurs=98 -> {n_98} | max={train_df[col].max()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, late_cols):
    sns.countplot(x=train_df[col], ax=ax, color="#8172B2")
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

## 5. Distribution de chaque variable selon la cible

Pour chaque variable explicative, on compare sa distribution entre les clients en défaut
(`SeriousDlqin2yrs` = 1) et les autres (= 0).

### Conclusion :


In [ ]:
feature_cols = [c for c in train_df.columns if c != TARGET]

n_cols = 3
n_rows = int(np.ceil(len(feature_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for ax, col in zip(axes, feature_cols):
    sns.boxplot(x=TARGET, y=col, data=train_df, ax=ax, showfliers=False)
    ax.set_title(col)
    ax.set_xlabel("SeriousDlqin2yrs")

# Cacher les axes en trop
for ax in axes[len(feature_cols):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 6. Corrélations et variables les plus liées à la cible

Corrélation de Pearson entre chaque variable numérique et la cible. À interpréter avec prudence :
la cible est binaire et plusieurs variables ont des valeurs manquantes/aberrantes non traitées
à ce stade — cette étape sert à orienter l'EDA, pas à conclure définitivement sur l'importance
des variables (cela viendra avec la modélisation).

### Conclusion :


In [ ]:
corr_matrix = train_df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Matrice de corrélation (train)")
plt.tight_layout()
plt.show()

In [ ]:
corr_with_target = (
    corr_matrix[TARGET]
    .drop(TARGET)
    .sort_values(key=lambda s: s.abs(), ascending=False)
)
corr_with_target